In [ ]:
# Cellule 1 -- Setup : clone du depot, sys.path, import des modules du projet.
# A executer tel quel dans Colab (le depot n'est pas deja present) ;
# en local, le clone est saute si le dossier existe deja.
import os
import sys
import subprocess

REPO_URL = "https://github.com/herverenard147/copilot_verification.git"
REPO_DIR = "copilot_verification"

if os.path.basename(os.getcwd()) != REPO_DIR and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)

if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)

sys.path.insert(0, os.getcwd())

from src.extractor import extract
from src.receipt import Receipt
from src.rules import audit, TAX_RATES

print("Modules importes depuis :", os.getcwd())
print("Pays geres par le moteur de regles :", TAX_RATES)

In [ ]:
# Cellule 2 -- Chargement de Donut (meme modele et memes appels que dans
# src/extractor.py et app.py : naver-clova-ix/donut-base-finetuned-cord-v2,
# utilise TEL QUEL, sans fine-tuning).
import torch
from transformers import DonutProcessor, VisionEncoderDecoderModel

MODEL_NAME = "naver-clova-ix/donut-base-finetuned-cord-v2"

processor = DonutProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Donut charge sur {device}")

In [ ]:
# Cellule 3 -- Fonction de test sur UNE image locale (un vrai ticket ivoirien,
# hors distribution CORD). Affiche image + JSON predit + audits CI et ID
# cote a cote, puis un commentaire honnete sur ce qui a marche ou rate.
import json
import os
from PIL import Image
import matplotlib.pyplot as plt


def test_on_local_image(path):
    """Teste Donut sur un ticket local et confronte le resultat au moteur
    de regles avec les deux configurations pays geres par l'app (CI 18%,
    ID 11%). Ne fine-tune rien : on observe Donut hors de sa distribution
    d'entrainement (CORD = tickets indonesiens uniquement)."""
    image = Image.open(path).convert("RGB")

    prediction = extract(image, model, processor, device)
    receipt = Receipt.from_gt_parse(prediction)

    audit_ci = audit(receipt, country="CI")
    audit_id = audit(receipt, country="ID")

    fig, ax = plt.subplots(figsize=(5, 7))
    ax.imshow(image)
    ax.axis("off")
    ax.set_title(os.path.basename(path))
    plt.show()

    print("--- JSON predit par Donut ---")
    print(json.dumps(prediction, indent=2, ensure_ascii=False))

    print("\n--- Recu reconstruit (src.receipt.Receipt) ---")
    print(receipt)

    print("\n--- Audit country=\"CI\" (TVA 18%) ---")
    for k, v in audit_ci.items():
        print(f"  {k}: {v}")

    print("\n--- Audit country=\"ID\" (TVA 11%) ---")
    for k, v in audit_id.items():
        print(f"  {k}: {v}")

    print("\n--- Commentaire honnete ---")
    if not prediction or not receipt.items:
        print("Extraction quasi vide : Donut n'a probablement pas reconnu la mise en page "
              "d'un ticket ivoirien (police, disposition, logos differents de CORD indonesien).")
    else:
        print(f"{len(receipt.items)} article(s) extrait(s) par Donut.")
    if receipt.total is None:
        print("Total non detecte.")
    if audit_ci["anomaly"] or audit_id["anomaly"]:
        print("Au moins un controle a echoue (voir le detail des audits ci-dessus) -- "
              "cela ne veut pas dire que Donut s'est trompe, seulement qu'un des trois "
              "montants ne colle pas arithmetiquement.")
    else:
        print("Aucune anomalie arithmetique detectee -- mais cela ne garantit pas "
              "l'exactitude des champs : un controle visuel du JSON reste indispensable.")
    print("Rappel : marchand et date ne sont JAMAIS extraits par ce pipeline (absents des "
          "labels CORD, donc jamais appris par Donut) -- voir src/llm.py pour l'extraction "
          "zero-shot par prompting.")

    return prediction, receipt, audit_ci, audit_id


# Exemple d'appel (remplacer par le chemin d'une vraie photo de ticket ivoirien) :
# test_on_local_image("mon_ticket_abidjan.jpg")

## Test sur un ticket de caisse ivoirien reel

### Biais geographique du modele

Donut (`naver-clova-ix/donut-base-finetuned-cord-v2`) a ete fine-tune exclusivement sur **CORD**, un
jeu de ~1000 tickets de caisse **indonesiens**. Le modele a donc appris les conventions visuelles et
linguistiques de ce contexte precis : mise en page des supermarches/restaurants indonesiens, mots
indonesiens (« Rp », noms de plats en bahasa), et un format de nombres ou la virgule/le point separe
les milliers (« 25,000 » = 25000).

Un ticket ivoirien differe sur plusieurs points previsibles : la devise (FCFA au lieu de Rp), la
langue (francais), une mise en page potentiellement differente (papier thermique, polices, logos
d'enseignes locales), et un taux de TVA different (18% en Cote d'Ivoire contre 11% en Indonesie dans
notre configuration `TAX_RATES`).

### Ce que Donut a rate sur ce ticket

[XX -- a completer apres le test sur une vraie photo : lister precisement les champs mal extraits,
ex. articles manquants, montants tronques, langue non reconnue, mise en page qui perturbe le modele]

Concretement, le JSON produit par `extract()` [XX -- coller ici un resume bref du JSON obtenu, ou
noter s'il est reste quasi vide].

### Pourquoi le moteur de regles fonctionne quand meme avec `country="CI"`

Le moteur de regles (`src/rules.py`) est **independant du modele d'extraction** : il ne fait aucune
hypothese sur la provenance des chiffres, seulement sur leur **coherence arithmetique** (somme des
lignes = sous-total, sous-total + taxe = total, taux de taxe plausible pour le pays choisi). Passer
`country="CI"` change uniquement le taux de reference utilise par `check_tax_rate` (18% au lieu de
11%) -- la logique a trois etats (`True`/`False`/`None`) reste identique.

Concretement, cela veut dire que **meme si Donut extrait mal certains champs**, le moteur de regles
reste capable de signaler une incoherence (`False`) ou une absence d'information (`None`) sur les
chiffres qu'il recoit, quels qu'ils soient -- la verification comptable ne depend pas de la qualite de
l'OCR, seulement de la disponibilite des trois montants (sous-total, taxe, total).

### Implications pour un deploiement reel

Ce test suggere que deployer ce pipeline sur des tickets ivoiriens sans adaptation reviendrait a
utiliser Donut en dehors de sa distribution d'entrainement : [XX -- preciser, une fois le test fait,
si l'extraction est totalement inexploitable, partiellement exploitable, ou etonnamment correcte].

Dans tous les cas, trois pistes se degagent pour un deploiement reel en Cote d'Ivoire :

1. **Fine-tuner Donut sur un petit jeu de tickets ivoiriens annotes** (meme quelques centaines de
   tickets suffiraient probablement a corriger les erreurs les plus systematiques de mise en
   page/langue).
2. **Garder le moteur de regles tel quel** : il reste la ligne de defense la plus fiable, puisqu'il ne
   depend d'aucune hypothese culturelle -- seulement de l'arithmetique.
3. **Rendre le drapeau "a verifier" plus visible** pour les champs extraits hors distribution : sur
   des tickets non-CORD, la probabilite d'erreur silencieuse est structurellement plus elevee, ce qui
   renforce la decision produit de ne jamais afficher de pourcentage de confiance trompeur (voir
   l'onglet Technique de `app.py` et `DESIGN.md`).

[XX -- ajouter ici, une fois le test realise, une conclusion en une phrase : le pipeline est-il
utilisable en l'etat sur des tickets ivoiriens, ou seulement comme point de depart ?]